# Allgemeine Analyse: Kongruenz nach Akteur

Erstes Auswertungs-Notebook: **Wie gut stimmen Parolen von Bundesrat, Parlament und Parteien mit dem Volksresultat überein?**

**Was passiert hier?**
- Datensatz mit Kongruenzwerten laden
- Boxplots: Parole (Ja/Nein/…) vs. Kongruenzwert pro Akteur
- Vergleich aller Akteure in einem Plot
- Blog-Grafik `d2_kongruenz_akteur.png` exportieren

**Datengrundlage**
- `data/processed/df_with_positions.csv` (Output von `2_berechnung.ipynb`, Spalten `zustimmung_*`)

**Vorher ausführen**
- `1_data_wrangling.ipynb` → `2_berechnung.ipynb`

**Danach**
- Kein Pflicht-Schritt; parallel: `3b_zeitliche_analyse`, `3c_*`, `3d_thematisch_analyse`
- Blog-Abschnitt „Allgemeine Erkenntnisse“ nutzt den exportierten Plot



# Setup
Projekt-Plots laden; `autoreload` aktualisiert `visualisierungen.py` bei Änderungen.

In [ ]:
%load_ext autoreload
%autoreload 2
from visualisierungen import *


## Daten laden

Kongruenz-Datensatz aus der Pipeline.


Datensatz mit Kongruenzwerten einlesen.


In [ ]:
df = pd.read_csv("../data/processed/df_with_positions.csv")
print(df.head())


Anzahl Abstimmungen mit echtem Volksresultat (Ja-Anteil ≠ 0).


In [ ]:
# Wie viele Abstimmungen haben effektiv stattgefunden?
print(len(df[df["volkja-proz"] != 0]))


# Explorative Analyse: Einzelne Akteure

Stimmt das Volk eher, wenn eine Institution **dafür** oder **dagegen** war? Pro Akteur ein Boxplot.


## Bundesrat

In [ ]:
# Boxplot Bundesrat: Parole vs. Kongruenz.
boxplot(df, df["br-pos_label"], df["zustimmung_br-pos"], titel="", xlabel="Bundesratsposition", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)


## Bundesversammlung

In [ ]:
# Boxplot Bundesversammlung
boxplot(df, df["bv-pos_label"], df["zustimmung_bv-pos"], titel="", xlabel="Postition Bundesversammlung", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)


## SP

In [ ]:
# Boxplot SP
boxplot(df, df["p-sps_label"], df["zustimmung_p-sps"], titel="", xlabel="Position SP", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

### Grüne

In [ ]:
# Boxplot Grüne
boxplot(df, df["p-gps_label"], df["zustimmung_p-gps"], titel="", xlabel="Position Grüne", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

## FDP

In [ ]:
# Boxplot FDP
boxplot(df, df["p-fdp_label"], df["zustimmung_p-fdp"], titel="", xlabel="Position FDP", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

## SVP

In [ ]:
# Boxplot SVP
boxplot(df, df["p-svp_label"], df["zustimmung_p-svp"], titel="", xlabel="Position SVP", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

# Final Plot D2: Vergleich aller Akteure

Kongruenz aller Institutionen und Parteien in einem Plot.

In [ ]:
# Alle Akteure in einem Boxplot nebeneinander vergleichen und Plot für den Blog speichern.
akteur_map = {
    'zustimmung_br-pos':   'Bundesrat',
    'zustimmung_bv-pos':   'Bundes-\nversammlung',
    'zustimmung_p-sps':    'SP',
    'zustimmung_p-gps':    'Grüne',
    'zustimmung_p-mitte':  'Mitte',
    'zustimmung_p-fdp':    'FDP',
    'zustimmung_p-svp':    'SVP',
}

akteur_cols = list(akteur_map.keys())

df_long = df[akteur_cols].melt(var_name="akteur", value_name="zustimmung")
df_long['akteur'] = df_long['akteur'].map(akteur_map)

# Optional: feste Reihenfolge für den Plot
df_long['akteur'] = pd.Categorical(df_long['akteur'], categories=akteur_map.values(), ordered=True)

fig = boxplot(df_long, x="akteur", y="zustimmung",
        xlabel="Akteur", ylabel="Kongruenzwert",
        palette=PALETTE_KATEGORIAL_VIELE_WERTE[:8],
        figsize=(8, 4.5),
        width=0.75)

plt.savefig("../Blog/blog_plots/d2_kongruenz_akteur.png",
            dpi=150, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Mittlere Kongruenz pro Akteur als Tabelle damit Blogtext verfasst werden kann.
df[akteur_cols].mean().rename(index=akteur_map).round(3).to_frame('mean_kongruenz')
